# آزمایش مستقل PPO موازی — ۴۰ میلیون گام محیطی\n\nاین نوت‌بوک یک آزمایش پژوهشی جدا از زنجیرهٔ تولیدی پنج‌مرحله‌ای CPU است. یک سیاست مشترک PPO را با ۴۰ محیط موازی و بودجهٔ مجموع ۴۰٬۰۰۰٬۰۰۰ انتقال آموزش می‌دهد؛ این تنظیمات، تنظیمات تولیدی ریپو نیستند. خط لولهٔ اصلی پنج‌مرحله‌ای در `.github/workflows/ppo-cpu-two-stage.yml` اجرا می‌شود و از دو Seed مرحلهٔ اول ۴۲ و ۴۰ و چهار محیط برای هر فرایند استفاده می‌کند.\n\nمحدودیت: این فقط آموزش شبیه‌سازی است؛ مدل فیزیکی واقعی و هم‌ارزی عددی کامل هنوز تأیید نشده‌اند و مدل برای پرواز واقعی آماده یا تأییدشده نیست. ۴۰ پردازه ممکن است از RAM/CPU نشست Colab بیشتر باشد.

In [ ]:
!git clone -b main https://github.com/samanramezani1377-hub/3d-fly-test.git\n%cd /content/3d-fly-test\n!python -m pip install -r training/requirements.txt\n

## ساخت هستهٔ فیزیک C++

اگر این مرحله خطا داد، آموزش را اجرا نکن و متن خطا را بفرست.

In [ ]:
!python training/build_native_core.py


## اجرای آزمایش مستقل ۴۰ میلیون گامی\n\nاین سلول عمداً تنظیمات آزمایشی ۴۰ محیطی را اجرا می‌کند و به Workflow پنج‌مرحله‌ای تولیدی متصل نیست. تعداد ۴۰ میلیون، مجموع انتقال‌های همهٔ محیط‌هاست؛ نه ۴۰ میلیون برای هر محیط. اگر منابع کافی نبود، `N_ENVS` را کاهش بده.

In [ ]:
TIMESTEPS = 40_000_000
N_ENVS = 40
N_STEPS = 1_000
BATCH_SIZE = 4_000
RUN_NAME = 'ppo-hover-parallel-40m'
!python training/train_ppo.py --timesteps {TIMESTEPS} --n-envs {N_ENVS} --n-steps {N_STEPS} --batch-size {BATCH_SIZE} --seed 42 --run-name {RUN_NAME} --save-freq 400000 --eval-freq 400000 --eval-episodes 10


## ارزیابی مدل نهایی

این ارزیابی بعد از پایان آموزش اجرا می‌شود و نتیجه را داخل پوشهٔ اجرا ذخیره می‌کند.

In [ ]:
!python training/evaluate.py --model training/runs/{RUN_NAME}/final_model.zip --episodes 10 --output training/runs/{RUN_NAME}/final-evaluation.json
!if [ -f training/runs/{RUN_NAME}/best/best_model.zip ]; then python training/evaluate.py --model training/runs/{RUN_NAME}/best/best_model.zip --episodes 10 --output training/runs/{RUN_NAME}/best-evaluation.json; fi


## دانلود همهٔ خروجی‌ها

ZIP شامل مدل نهایی، بهترین مدل ارزیابی‌شده، checkpointها، لاگ‌ها و گزارش‌های ارزیابی است. آن را قبل از بستن Colab دانلود کن.

In [ ]:
!cd training/runs && zip -qr /content/{RUN_NAME}-results.zip {RUN_NAME}
from google.colab import files
files.download(f'/content/{RUN_NAME}-results.zip')


## نکتهٔ مهم برای ادامهٔ کار

۴۰ میلیون گام آموزش، موفقیت را تضمین نمی‌کند. معیار تصمیم، گزارش ارزیابی، نرخ برخورد، موفقیت هاور، سهم زمان پایدار، خطای ارتفاع و سرعت عمودی است. اگر نتیجه خوب نبود، ZIP را در همین گفتگو بفرست تا مرحلهٔ بعد را بر اساس گزارش‌ها اصلاح کنیم. این مدل را روی کوادکوپتر واقعی اجرا نکن.